# COGS 188 - Project Proposal

# Names

- Kaden Leung
- Khristian-Rey Tamoria
- Justin Tann

Repository: https://github.com/kaden-leung/COGS188_group_project

# Abstract 
Product teams usually choose between design variants with A/B tests, which split traffic evenly until the test ends and keep showing losing variants to users. Bandit algorithms shift traffic toward variants that perform well as evidence accumulates, possibly at the cost of being less sure which variant is truly best. We will measure this trade-off using the Open Bandit Dataset: about 13.5 million logged fashion recommendations from ZOZOTOWN, each recording the user's features, the item shown (one of 80, in one of 3 positions), and whether the user clicked. From the uniformly random logs we will fit a click model, selecting user features by cross-validation, and use it to simulate users; we will check this simulator against the click rate ZOZOTOWN's production Thompson sampling policy actually achieved. In simulation we will compare a fixed-horizon A/B test, context-free bandits (ε-greedy, optimistic initialization, gradient bandit, UCB, Thompson sampling), and a contextual bandit (LinUCB), tuning each method's hyperparameters. We will also test what happens when shoppers' tastes change during the experiment, comparing bandits that keep all past data with bandits that discount it. Success will be measured by regret, the probability of identifying the best item, item-exposure concentration, and computation time.

# Background

Fill in the background and discuss the kind of prior work that has gone on in this research area here. **Use inline citation** to specify which references support which statements.  You can do that through HTML footnotes (demonstrated here). I used to reccommend Markdown footnotes (google is your friend) because they are simpler but recently I have had some problems with them working for me whereas HTML ones always work so far. So use the method that works for you, but do use inline citations.

Here is an example of inline citation. After government genocide in the 20th century, real birds were replaced with surveillance drones designed to look just like birds<a name="lorenz"></a>[<sup>[1]</sup>](#lorenznote). Use a minimum of 3 to 5 citations, but we prefer more <a name="admonish"></a>[<sup>[2]</sup>](#admonishnote). You need enough citations to fully explain and back up important facts. 

Remeber you are trying to explain why someone would want to answer your question or why your hypothesis is in the form that you've stated. 

# Problem Statement

A platform must decide which items to recommend to each user it sees. At each impression $t = 1, \dots, T$, a user with features $x_t$ arrives, and the agent fills 3 display positions with distinct items chosen from $K = 80$, so there are $K(K-1)(K-2) = 492{,}960$ possible ordered recommendations. The user clicks the item $a$ shown in position $p$ with an unknown probability $\mu(x_t, a, p)$. Following the assumption made by the dataset's authors, this probability depends only on the user, the item, and its position, not on the other items shown. The reward $R_t$ is the number of clicks at impression $t$, so its expected value is $\sum_{p=1}^{3} \mu(x_t, A_{t,p}, p)$, where $A_{t,p}$ is the item placed in position $p$. The best policy $\pi^*$ shows each user the recommendation with the highest expected number of clicks.

The agent has two goals that pull against each other:

1. **Earn:** maximize expected clicks during the experiment, or equivalently minimize the expected regret
$$\text{Regret}(T) = \mathbb{E}\left[\sum_{t=1}^{T} \big(R_t^{\pi^*} - R_t\big)\right],$$
where $R_t^{\pi^*}$ is the reward the best policy would have earned for the same user.
2. **Learn:** at the end of the experiment, recommend the item that is truly best on average, $a^* = \arg\max_a \mathbb{E}_x[\mu(x, a, 1)]$; that is, maximize $P(\hat{a} = a^*)$.

A standard A/B test favors learning: it splits traffic evenly until the end, so it pays for exploration on every losing item. Bandit algorithms favor earning by shifting traffic toward items that look better, which may make their final recommendation less reliable and may concentrate exposure on a few items. Our problem is to quantify these trade-offs for each method as the number of items $K$, the horizon $T$, and the speed at which tastes change vary.

**Tastes can change during the experiment.** Fashion demand shifts over time, so the click probability can depend on when the impression happens, $\mu_t(x, a, p)$. "Best" then splits in two: the item that is best on average over the experiment, $a^*_{\text{avg}} = \arg\max_a \frac{1}{T}\sum_{t=1}^{T} \mathbb{E}_x[\mu_t(x, a, 1)]$, and the item that is best at the end, $a^*_{\text{now}} = \arg\max_a \mathbb{E}_x[\mu_T(x, a, 1)]$. A bandit that keeps all of its data can stay on an early winner after it fades and misjudge both; a bandit that discounts old data adapts faster but answers only the "best now" question. In a small synthetic pilot with made-up click rates (three items whose popularity shifts weekly; `experiments/drift_ab_vs_bandit.py` in our repository), Thompson sampling earned 6.5% more clicks than an even three-way test but named $a^*_{\text{avg}}$ in only 63% of 1,000 simulated months, versus 97% for the test. We will test whether this holds with real ZOZOTOWN click rates, using the day-to-day changes across the dataset's 7 days of logs and drift scenarios built from its real item click rates.

This is not the 10-armed testbed from class, where each arm's value is drawn from $\mathcal{N}(0,1)$ and rewards are normally distributed around it with unit variance. Here rewards are binary, rare, and depend on who the user is, which makes the problem hard for simple methods and infeasible to solve by brute force:
- **Simple policies leave clicks on the table.** In the dataset, recommending items uniformly at random earned a 0.35% click-through rate, while ZOZOTOWN's production Thompson sampling policy earned 0.50%, about 43% more.
- **Rewards are rare.** By the common rule of thumb $n \approx 16\,p(1-p)/\delta^2$ (two-sided $\alpha = 0.05$, 80% power), detecting a difference of $\delta = 0.05$ percentage points at $p = 0.35\%$ needs roughly 220,000 impressions per item, or about 18 million for all 80 items. That is more than the 13.5 million impressions logged in this campaign.
- **Greedy methods can lock onto the wrong item.** With so few clicks, early estimates are noisy, and an agent that always exploits its current estimate can commit to a suboptimal item permanently.
- **The optimal policy is intractable at scale.** With Beta priors, the Bayes-optimal policy for the simpler non-contextual problem can be computed by dynamic programming over posterior states, but the number of states grows on the order of $T^{2K}$. Even $K = 2$ items over $T = 100$ impressions gives about 4.6 million states, and $K = 2$ over $T = 1{,}000$ gives about $4.2 \times 10^{10}$.

The problem is **quantifiable** (regret and $P(\hat{a} = a^*)$ are defined above), **measurable** (every simulated impression produces an observable click or no-click, so both quantities can be computed for each run), and **replicable** (the simulator is built from a public dataset, and each experiment is repeated over many independent random seeds).

# Data

You should have a strong idea of what dataset(s) will be used to accomplish this project. 

If you know what (some) of the data you will use, please give the following information for each dataset:
- link/reference to obtain it
- description of the size of the dataset (# of variables, # of observations)
- what an observation consists of
- what some critical variables are, how they are represented
- any special handling, transformations, cleaning, etc will be needed

If you don't yet know what your dataset(s) will be, you should describe what you desire in terms of the above bullets.

# Proposed Solution

In this section, clearly describe a solution to the problem. The solution should be applicable to the project domain and appropriate for the dataset(s) or input(s) given. Provide enough detail (e.g., algorithmic description and/or theoretical properties) to convince us that your solution is applicable. Why might your solution work? Make sure to describe how the solution will be tested.  

If you know details already, describe how (e.g., library used, function calls) you plan to implement the solution in a way that is reproducible.

If it is appropriate to the problem statement, describe a benchmark model<a name="sota"></a>[<sup>[3]</sup>](#sotanote) against which your solution will be compared. 

# Evaluation Metrics

Propose at least one evaluation metric that can be used to quantify the performance of both the benchmark model and the solution model. The evaluation metric(s) you propose should be appropriate given the context of the data, the problem statement, and the intended solution. Describe how the evaluation metric(s) are derived and provide an example of their mathematical representations (if applicable). Complex evaluation metrics should be clearly defined and quantifiable (can be expressed in mathematical or logical terms).

# Ethics & Privacy

If your project has obvious potential concerns with ethics or data privacy discuss that here.  Almost every ML project put into production can have ethical implications if you use your imagination. Use your imagination. Get creative!

Even if you can't come up with an obvious ethical concern that should be addressed, you should know that a large number of ML projects that go into producation have unintended consequences and ethical problems once in production. How will your team address these issues?

Consider a tool to help you address the potential issues such as https://deon.drivendata.org

# Team Expectations 

Put things here that cement how you will interact/communicate as a team, how you will handle conflict and difficulty, how you will handle making decisions and setting goals/schedule, how much work you expect from each other, how you will handle deadlines, etc...
* *Team Expectation 1*
* *Team Expectation 2*
* *Team Expecation 3*
* ...

# Project Timeline Proposal

| Meeting Date  | Meeting Time | Completed Before Meeting  | Discuss at Meeting |
|---|---|---|---|
| 10/27 (Tue) | TBD | Read the project brief and skim the Open Bandit Dataset paper (all) | Confirm topic and design; confirm proposal section owners; agree on shared notation for the Problem Statement and Evaluation Metrics |
| 10/29 (Thu) | TBD | Draft proposal sections on individual branches (Kaden: Names, Abstract, Problem Statement, Timeline; Khristian-Rey and Justin: remaining sections); share the synthetic drift pilot's figure as the prototype for the Proposed Solution (Kaden); confirm project scope with Prof. Fleischer in office hours (Kaden) | Merge branches through pull requests; edit the proposal for consistency |
| 10/30 (Fri) | Before 11:59 PM | Final proofread (all) | Submit proposal on Gradescope |
| 11/2 (Mon) | TBD | Download the full dataset and add a download script to the repo; compute click-through rates by item and position, and measure how much they change across the 7 logged days (Khristian-Rey) | Review exploratory analysis and proposal feedback; finalize the simulator design, including drift scenarios |
| 11/9 (Mon) | TBD | Fit the click model, select user features by cross-validation, and validate the simulator against the production Thompson sampling click rate (Khristian-Rey); implement the A/B test, context-free bandits, LinUCB, and bandits that discount old data (discounted Thompson sampling, constant step-size ε-greedy) with sanity tests (Justin); size the A/B test with a power analysis and implement the exposure-concentration metric (Kaden) | Review validation results; choose horizons, update batch size, and hyperparameter grid |
| 11/16 (Mon) | TBD | Run main experiments: A/B test vs. bandits across numbers of items $K$ (Justin); textbook vs. real-data parameter study (Khristian-Rey); earn-vs-learn analysis, drift experiments comparing "best on average" with "best now", and statistical comparisons across seeds (Kaden) | Review results; decide on optional extensions (dynamic programming study, off-policy evaluation check) |
| 11/23 (Mon) | TBD | Optional extensions if chosen (dynamic programming: Justin; off-policy evaluation: Khristian-Rey); draft Background, Data, and Methods sections (section owners) | Thanksgiving week: review drafts asynchronously if we cannot meet |
| 11/30 (Mon) | TBD | Draft Results, Discussion, and Conclusion (Kaden leads); clean up the repository so every figure reproduces from one command (all) | Full read-through of the report; assign final edits |
| Finals week (date TBD) | Before deadline | Final edits; export PDF; confirm the repo link in the report (all) | Submit final report |

# Footnotes
<a name="lorenznote"></a>1.[^](#lorenz): Lorenz, T. (9 Dec 2021) Birds Aren’t Real, or Are They? Inside a Gen Z Conspiracy Theory. *The New York Times*. https://www.nytimes.com/2021/12/09/technology/birds-arent-real-gen-z-misinformation.html<br> 
<a name="admonishnote"></a>2.[^](#admonish): Also refs should be important to the background, not some randomly chosen vaguely related stuff. Include a web link if possible in refs as above.<br>
<a name="sotanote"></a>3.[^](#sota): Perhaps the current state of the art solution such as you see on [Papers with code](https://paperswithcode.com/sota). Or maybe not SOTA, but rather a standard textbook/Kaggle solution to this kind of problem
